# 0305 · Guías de la reserva (septiembre a noviembre), con las mismas reglas de la base v4

**Fase 4 (prueba única).** Mismo cuaderno que `0305_GuiasConAbiertas`, apuntando a `RESERVA_POSTERIOR/`. Escribe `RESERVA_POSTERIOR/LOG_GUIAS/RAW_V4/LOG_GUIAS_BOGOTA_V4.parquet`, con las mismas columnas, la misma minimización de datos, la misma lista de estados (1 a 5 y 9) y la misma sal del cliente (`SEGURIDAD/sal_cliente.txt`), para que la historia por cliente de la base de desarrollo aplique a la reserva.

**Guarda.** Solo corre si existe `V4/P4b_decision_v2.json`.

In [0]:
%pip install "duckdb<2"

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 119.0 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os, secrets, duckdb, pandas as pd
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
RUTA_BATCH_LOG_GUIAS = f"{PATH_DATABRICKS}RESERVA_POSTERIOR/LOG_GUIAS/LOG_GUIAS*.parquet"  # reserva (crudos)
RUTA_GUIAS_V3        = f"{PATH_DATABRICKS}LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet"           # 0300 (no se toca)
RUTA_GUIAS_V4        = f"{PATH_DATABRICKS}RESERVA_POSTERIOR/LOG_GUIAS/RAW_V4/LOG_GUIAS_BOGOTA_V4.parquet"   # salida
RUTA_EST_ENVIO       = f"{PATH_DATABRICKS}PARAMETRICAS/PAR.ESTADOSENVIO.parquet"
RUTA_SAL_CLIENTE     = f"{PATH_DATABRICKS}SEGURIDAD/sal_cliente.txt"

# Fijar con P10_1c_guias_mes_estado.csv (0345). Propuesta del director: 1 a 5
# (1 recibido, 2 en procesamiento, 3 entregado, 4 entregado a remitente, 5 siniestrado).
# 9 (en alistamiento del cliente) se mantiene porque 0300 (>= 3) ya lo incluía: así el único cambio frente a la v3
# es sumar las abiertas 1 y 2. Las que no tengan salida a reparto ni admisión válida salen en 0330 (sin T0), con conteo.
ESTADOS_ENVIO_INCLUIDOS = [1, 2, 3, 4, 5, 9]
ESTADOS_EXCLUIDOS       = {}          # {id: "razón"} para estados que aparezcan y no deban entrar (p. ej., anulada)
# Las guías con ID_ESTADOENVIO nulo no tienen estado: se excluyen siempre y se cuentan en el control previo.

In [0]:
# GUARDA (C2): la reserva solo se abre con la decisión v2 ya registrada
import os, json
RUTA_DECISION = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/P4b_decision_v2.json"
assert os.path.exists(RUTA_DECISION), "No hay P4b_decision_v2.json: la reserva no se abre antes de registrar la decisión"
print("Decisión registrada el", json.load(open(RUTA_DECISION))["fecha_registro"], "· se puede construir la base de la reserva")

Decisión registrada el 2026-11-08T00:46:23-05:00 · se puede construir la base de la reserva


In [0]:
# ESCRITURA EN EL VOLUME · en Databricks serverless el FUSE de /Volumes no acepta las escrituras
# asíncronas de DuckDB ("Async write failed ... errno 5"): el COPY escribe en /tmp y luego copia.
import os, re, shutil, tempfile, uuid
def copy_a_volume(sql_copy, con=None):
    """Ejecuta un COPY (...) TO '<ruta>' de DuckDB escribiendo en /tmp y copiando el resultado a <ruta>."""
    destino_m = list(re.finditer(r"\bTO\s+'([^']+)'", sql_copy, flags=re.IGNORECASE))[-1]
    destino = destino_m.group(1)
    local = os.path.join(tempfile.gettempdir(), f"duckdb_{uuid.uuid4().hex}_{os.path.basename(destino)}")
    sql_local = sql_copy[:destino_m.start(1)] + local + sql_copy[destino_m.end(1):]
    try:
        (con or duckdb).execute(sql_local)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        try:
            shutil.copyfile(local, destino)
        except OSError:
            dbutils.fs.cp(f"file:{local}", destino)
    finally:
        if os.path.exists(local):
            os.remove(local)
    print(f"Escrito en {destino}")
    return destino

In [0]:
# Control previo: todo estado presente debe estar incluido o excluido con razón
presentes = duckdb.sql(f"""SELECT G.ID_ESTADOENVIO, any_value(E.NOMBRE_ESTADOENVIO) AS nombre, COUNT(*) AS guias
                          FROM read_parquet('{RUTA_BATCH_LOG_GUIAS}') G
                          LEFT JOIN read_parquet('{RUTA_EST_ENVIO}') E USING (ID_ESTADOENVIO)
                          WHERE G.ID_CIUDAD_DESTINO = 10 AND length(G.NUMERO_GUIA::VARCHAR) >= 8
                          GROUP BY 1 ORDER BY 1""").df()
display(presentes)
nulos = int(presentes.loc[presentes.ID_ESTADOENVIO.isna(), "guias"].sum())
print(f"Guías con estado nulo (se excluyen): {nulos:,}")
sin_decidir = set(presentes.ID_ESTADOENVIO.dropna().astype(int)) - set(ESTADOS_ENVIO_INCLUIDOS) - set(ESTADOS_EXCLUIDOS)
assert not sin_decidir, f"Estados presentes sin decidir: {sorted(sin_decidir)}. Agregarlos a ESTADOS_ENVIO_INCLUIDOS o a ESTADOS_EXCLUIDOS"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ID_ESTADOENVIO,nombre,guias
1.0,RECIBIDO DEL CLIENTE,8344
2.0,EN PROCESAMIENTO,27450
3.0,ENTREGADO,718326
4.0,ENTREGADO A REMITENTE,16003
5.0,SINIESTRADO,138
9.0,EN ALISTAMIENTO DEL CLIENTE,62652
null,null,556


Guías con estado nulo (se excluyen): 556


In [0]:
def leer_o_crear_sal(ruta):
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        with open(ruta, "w") as f: f.write(secrets.token_hex(16))
        print(f"Sal nueva creada en {ruta}: los seudónimos de cliente no coincidirán con los de la v3.")
    with open(ruta) as f: sal = f.read().strip()
    assert len(sal) >= 16 and "'" not in sal, "La sal debe tener al menos 16 caracteres y no llevar comillas"
    return sal

_sal_cliente = leer_o_crear_sal(RUTA_SAL_CLIENTE)
lista = ", ".join(str(int(x)) for x in ESTADOS_ENVIO_INCLUIDOS)
con = duckdb.connect()
copy_a_volume(f"""
    COPY (
        SELECT 
            NUMERO_GUIA,ID_TIPOGUIA,md5(CAST(ID_CLIENTE AS VARCHAR) || '{_sal_cliente}') AS ID_CLIENTE,ID_CANAL,ID_DIVISION_PROV,ID_1,ID_2,ID_PAIS_ORIGEN,ID_CIUDAD_ORIGEN,
            ID_PAIS_DESTINO,ID_CIUDAD_DESTINO,ID_TIPO_ZONA_URBA,ID_PAIS_ZONA_URB_ENTR,ID_CIUDAD_ZONA_URB_ENTR,ID_ZONA_URBA,ID_ESTADOENVIO,
            ID_ESTADOGUIA,ID_CARACT_ENVIO,ID_TIEMPO,ID_MEDTRANS,ID_PRODUCTO,ID_SUBPRODUCTO,ID_BODEPROV,ID_PAIS_UBICACION,ID_CIUDAD_UBICACION,
            ID_DIVISION_UBICACION,FECHA_ENVIO,NULL AS ID_REMITE, NULL AS ID_DESTINA,null as  NOMBRE_REMITE_GUIA,null as  APELLIDO_REMITE_GUIA,
            null as  NOMBRE_DESTINA_GUIA,null as  APELLIDO_DESTINA_GUIA,null as  TELEFONO_REMITE_GUIA,null as  TELEFONO_DESTINA_GUIA,
            null as  DIRECCION_REMITE_GUIA,null as  DIRECCION_DESTINA_GUIA,NULL AS IDEN_RECEP_SERVI, NULL AS IDEN_ENTR_SERVI, NULL AS CONTENIDO,LARGO_ENVIO,
            ALTO_ENVIO,ANCHO_ENVIO,LARGO_ENV_OPERA,ALTO_ENV_OPERA,ANCHO_ENV_OPERA,PESO_GUIA,PESO_FISICO_GUIA,PESO_VOLU_GUIA,
            PESO_FISICO_OPERA,PESO_VOLU_OPERA,PESO_FACTURADO,FECHA_ENTREGA,NUMERO_PIEZAS,CASE WHEN NULLIF(TRIM(CAST(IDEN_PERS_RECIBE AS VARCHAR)), '') IS NOT NULL THEN 'S' END AS IDEN_PERS_RECIBE, CASE WHEN NULLIF(TRIM(CAST(NOMB_PERS_RECIBE AS VARCHAR)), '') IS NOT NULL THEN 'S' END AS NOMB_PERS_RECIBE,
            ID_REMPLAZA,NULL AS IDEN_DEPOSITANTE, NULL AS NOMB_DEPOSITANTE,
            FECHA_ULTIMA_MOD,NULL AS DIRECCION_ESTANDAR, NULL AS OBSERVACIONES,GUIA_EXTERNA,ID_DOCU_ULT_MOVI,NU_DOCU_ULT_MOVI,CONS_DOCU_ULT_MOVI,ID_FACTURADO,VALOR_DECLARADO_GUIA,VALOR_SEG_MAN_GUIAS,VALOR_FLETES_GUIAS,VALOR_DESCUENTO_GUIAS,VALOR_PEDIDO_GUIAS,VALOR_SEG_MAN_GUIAS_FACT,VALOR_FLETES_GUIAS_FACT,ID_REQUIERE_EMBALAJE,null as  LOGIN_EMPLEADO,FECHA_CREACION,FECHA_MODIFICACION,FECHA_MODIFICACION_DG,FECHA_EXTRACCION_DW
        FROM read_parquet('{RUTA_BATCH_LOG_GUIAS}')
        WHERE ID_CIUDAD_DESTINO = 10
          AND LEN(NUMERO_GUIA::VARCHAR) >= 8
          AND ID_ESTADOENVIO IN ({lista})
        QUALIFY ROW_NUMBER() OVER (PARTITION BY NUMERO_GUIA ORDER BY FECHA_EXTRACCION_DW DESC, FECHA_ULTIMA_MOD DESC) = 1
    ) TO '{RUTA_GUIAS_V4}' (FORMAT PARQUET, COMPRESSION ZSTD)""", con)
con.close()
del _sal_cliente

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Escrito en /Volumes/proyectogrado/source/source/RESERVA_POSTERIOR/LOG_GUIAS/RAW_V4/LOG_GUIAS_BOGOTA_V4.parquet


In [0]:
# Controles: una fila por guía y conteo por mes y estado (solo conteos)
c = duckdb.sql(f"""SELECT COUNT(*) AS filas, COUNT(DISTINCT NUMERO_GUIA) AS guias, MIN(FECHA_ENVIO) AS primera_admision,
                          MAX(FECHA_ENVIO) AS ultima_admision, MAX(FECHA_EXTRACCION_DW) AS ultima_extraccion
                   FROM read_parquet('{RUTA_GUIAS_V4}')""").df()
display(c)
assert c.loc[0, "filas"] == c.loc[0, "guias"], "Hay guías repetidas en la salida"
display(duckdb.sql(f"""SELECT strftime(FECHA_ENVIO, '%Y-%m') AS mes, ID_ESTADOENVIO, COUNT(*) AS guias
                       FROM read_parquet('{RUTA_GUIAS_V4}') GROUP BY ALL ORDER BY ALL""").df()
       .pivot_table(index="mes", columns="ID_ESTADOENVIO", values="guias", fill_value=0))

filas,guias,primera_admision,ultima_admision,ultima_extraccion
832913,832913,2026-09-01T00:00:00.000Z,2026-11-06T15:00:00.000Z,2026-11-05T09:45:09.000Z


1.0,2.0,3.0,4.0,5.0,9.0
2977.0,1064.0,362358.0,9231.0,126.0,23587.0
3131.0,5325.0,343924.0,6768.0,12.0,26880.0
2236.0,21061.0,12044.0,4.0,0.0,12185.0


## Qué sigue

Con este archivo se corre `0330_BaseAnaliticaV4` (que lo lee en `RUTA_GUIAS`). No hace falta volver a correr el resto de 0300: 0330 lee los movimientos y documentos crudos.